# Review Notebook: For Loops & Local AI Models with dotenv

This notebook reviews using **for loops** to process text data using **local AI models** (Ollama). We will load a political manifesto dataset, run full-text prompts on a subset, perform sentence-level analysis using slicing, and save intermediate results to JSON to prevent data loss on interruption.

## 1. Setting up Environment Variables with dotenv

Before running code, we load configuration from our `.env` file using the `dotenv` package.

## 1.A Importing

In [17]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from the project's .env file
load_dotenv()
print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [18]:
os.getenv("API_GEMINI")

'HELLO'

## 1.1 Using in function

In [ ]:
gemini_api_key = os.getenv("API_GEMINI")


In [ ]:

def llm_gemini(input_text, gemini_model="gemini-3.5-flash"):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{gemini_model}:generateContent"
    response = requests.post(
        url,
        params={"key": gemini_api_key # OR PASTE THIS DIRECTLY IN FROM UP TOP: os.getenv("API_GEMINI")
                },
        json={"contents": [{"parts": [{"text": input_text}]}]},
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]

## 2. Setting Up Ollama (Local AI Model)

We use Ollama to run models locally on our computer. Below is a helper function to send prompts to Ollama.

In [19]:
OLLAMA_MODEL = "llama3.2:latest"

def llm_ollama(prompt, model=OLLAMA_MODEL):
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        },
        timeout=120
    )
    response.raise_for_status()
    return response.json()["message"]["content"]

## 3. Loading the Dataset

We load the English-speaking manifesto dataset from `data/examples/week_7/manifesto_english_speaking.parquet` using pandas.

In [21]:
# Load the parquet dataset
df = pd.read_parquet("../../data/examples/week_7/manifesto_english_speaking.parquet")
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

Loaded 381 rows and 5 columns.


party                   partyname          keys  \
countryname date                                                          
Australia   1961-12-01  63320      Australian Labor Party  63320_196112   
            1961-12-01  63330      Democratic Labor Party  63330_196112   
            1961-12-01  63620  Liberal Party of Australia  63620_196112   
            1961-12-01  63810               Country Party  63810_196112   
            1963-11-01  63320      Australian Labor Party  63320_196311   

                        manifesto_id  \
countryname date                       
Australia   1961-12-01  63320_196112   
            1961-12-01  63330_196112   
            1961-12-01  63620_196112   
            1961-12-01  63810_196112   
            1963-11-01  63320_196311   

                                                                     text  
countryname date                                                           
Australia   1961-12-01  Labour government would subsidise interest rat...  
            1961-12-01  New concepts needed for Nation’s problems.  Me...  
            1961-12-01  Good government pledges by Menzies, no long li...  
            1961-12-01  Pledge by McEwen to protect Industries.  SHEPP...  
            1963-11-01  CALWELL LISTS MAIN POLL ISSUES  13 KEY POINTS ...

## 4. Example 1: Full-Text Processing with Incremental JSON Saving

We select 5 rows from the dataset and run a for loop using Ollama on the full text of each. To prevent losing progress in case the execution is interrupted, we save each response to a JSON file incrementally. Finally, we load the JSON and put the results back into our DataFrame.

In [22]:
# Select 5 examples (rows) to process
df_subset = df.head(5).copy()

# Initialize results dictionary
full_text_results = {}
output_json_path = "data/temp/manifesto_full_text_results.json"

# Ensure the output directory exists
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

In [23]:
print("Starting full text processing...")
for idx, row in df_subset.iterrows():
    # Convert index (which is a MultiIndex tuple) to string for JSON compatibility
    idx_str = str(idx)
    
    prompt = f"Summarize the following political text in one short sentence: {row['text']}"
    
    print(f"Processing row: {idx_str[:60]}...")
    try:
        response = llm_ollama(prompt)
        print(response)
        full_text_results[idx_str] = response
    except Exception as e:
        print(f"Error processing row {idx_str}: {e}")
        full_text_results[idx_str] = f"Error: {e}"
        
    # Save incrementally to JSON to prevent loss on interruption
    with open(output_json_path, "w") as f:
        json.dump(full_text_results, f, indent=4)

print("Processing complete!")

Starting full text processing...
Processing row: ('Australia', Timestamp('1961-12-01 00:00:00'))...
A Labour government would offer interest rates of up to 3.5% for housing loans to encourage home-building by subsidizing loans on a maximum of 90,000 units per year.
Processing row: ('Australia', Timestamp('1961-12-01 00:00:00'))...
Here is a short summary of the text in one sentence:

The Democratic Labour Party's policy speech calls for vital new concepts to solve Australia's problems, including stronger defense, new markets, and more equitable social services, while criticizing the ALP as dominated by Communists.
Processing row: ('Australia', Timestamp('1961-12-01 00:00:00'))...
Here is a short summary of the political text in one sentence:

Australian Prime Minister Robert Menzies delivered a policy speech, promising to continue the country's economic growth and development, while criticizing the Australian Labor Party (ALP) as divided and lacking experience, and appealing for a majo

In [24]:
# Load the results back from the JSON file
with open(output_json_path, "r") as f:
    saved_results = json.load(f)

In [25]:

# Put the results back into the DataFrame
df_subset["summary"] = df_subset.index.map(lambda idx: saved_results.get(str(idx), None))
df_subset[["partyname", "summary"]].head()

partyname  \
countryname date                                     
Australia   1961-12-01      Australian Labor Party   
            1961-12-01      Democratic Labor Party   
            1961-12-01  Liberal Party of Australia   
            1961-12-01               Country Party   
            1963-11-01      Australian Labor Party   

                                                                  summary  
countryname date                                                           
Australia   1961-12-01  Australian Minister for Trade J. McEwen pledge...  
            1961-12-01  Australian Minister for Trade J. McEwen pledge...  
            1961-12-01  Australian Minister for Trade J. McEwen pledge...  
            1961-12-01  Australian Minister for Trade J. McEwen pledge...  
            1963-11-01  Here is a one-sentence summary of the text:\n\...

## 5. Example 2: Sentence-Level Processing with Slicing & Incremental JSON Saving

In this example, we take a single text, split it into sentences, and use slicing (e.g. `[:5]`) so we can easily scale the number of sentences up or down. We run Ollama on each sentence, save results to JSON incrementally, and load them back into a new DataFrame.

In [27]:
# Select the text of a single example
one_text = df.iloc[0]["text"]

# Split by sentence
sentences = [s.strip() for s in one_text.split(". ") if s.strip()]
print(f"Total sentences in the text: {len(sentences)}")
sentences[:3]

Total sentences in the text: 153


['Labour government would subsidise interest rates for housing to encourage home-building',
 'Mr Calwell said the Labour Party believed that home builders should pay no more tahn 3,5 per cent interest on their loans',
 'A Labour Government would ensure this rate of interest by subsidising loans on a maximum of 90000 housing units a year']

In [29]:
# Use slicing to choose a subset (e.g., first 5 sentences) to process
# You can scale this up by changing the slice range
sliced_sentences = sentences[:5]
print(f"Processing a slice of {len(sliced_sentences)} sentences...")

Processing a slice of 5 sentences...


In [30]:
sentence_results = {}
sentence_json_path = "data/temp/sentence_results.json"

for idx, sentence in enumerate(sliced_sentences):
    prompt = f"Identify the main topic of this sentence: {sentence}"
    print(f"Processing sentence {idx + 1}/{len(sliced_sentences)}...")
    
    try:
        response = llm_ollama(prompt)
        sentence_results[idx] = {
            "sentence": sentence,
            "topic": response
        }
    except Exception as e:
        print(f"Error processing sentence {idx}: {e}")
        sentence_results[idx] = {
            "sentence": sentence,
            "topic": f"Error: {e}"
        }
        
    # Save incrementally to JSON to prevent loss on interruption
    with open(sentence_json_path, "w") as f:
        json.dump(sentence_results, f, indent=4)

print("Sentence processing complete!")

Processing sentence 1/5...
Processing sentence 2/5...
Processing sentence 3/5...
Processing sentence 4/5...
Processing sentence 5/5...
Sentence processing complete!


In [31]:
# Load the sentence results back from the JSON file
with open(sentence_json_path, "r") as f:
    saved_sentence_results = json.load(f)

# Put the results back into a DataFrame
sentence_df = pd.DataFrame.from_dict(saved_sentence_results, orient="index")
sentence_df

,sentence,topic
0,Labour government would subsidise interest rat...,The main topic of this sentence is Housing and...
1,Mr Calwell said the Labour Party believed that...,The main topic of this sentence is the economi...
2,A Labour Government would ensure this rate of ...,"The main topic of this sentence is economics, ..."
3,The cost of this subsidy was expected to be ab...,The main topic of this sentence is: Subsidy.
4,SLUGGISH DEMAND,"The main topic of the sentence ""SLUGGISH DEMAN..."
